In [ ]:
import os, time, math, pprint, statistics as stats
from collections import defaultdict, deque
os.environ['MAVLINK20'] = '1'
from pymavlink import mavutil

# Normal px4 port
# master = mavutil.mavlink_connection('udp:0.0.0.0:14540')
# real attack Custom port
master = mavutil.mavlink_connection('udpin:0.0.0.0:17001')
# px4 custom port
# master = mavutil.mavlink_connection('udpin:0.0.0.0:17000')

master.wait_heartbeat()
print("Connection successful: System ID: %d, Component ID: %d" % (master.target_system, master.target_component))

In [ ]:
# Set/Get Gyro Bias test. Flushing the mavlink message queue.
t0 = time.monotonic()
master.mav.set_gyro_bias_send(0.0, 0.0, 0.0)

# Flush
flushed = 0
while True:
    m = master.recv_match(blocking=False)
    if not m:
        break
    flushed += 1
if flushed:
    print(f"Cleared {flushed} remaining messages")

message = master.recv_match(type="GET_GYRO_BIAS", blocking=True, timeout=1)
if message:
    pprint.pprint(message.to_dict())
print(f"elapsed time: {time.monotonic() - t0}")

In [ ]:
# Check what messages are being received for 5 seconds
received_messages = set()
timeout = time.time() + 5

print("Received MAVLink message types:")
while time.time() < timeout:
    msg = master.recv_match(blocking=False)
    if msg:
        received_messages.add(msg.get_type())

for msg_type in sorted(received_messages):
    print(f"- {msg_type}")

In [ ]:
MSG_NAMES = (
    "GPS_RAW_INT",
    "ATTITUDE_QUATERNION",
    "SCALED_IMU",
    "GET_GYRO_BIAS",
)

def measure_rates(duration_s=15.0, msg_names=MSG_NAMES, verbose_every=0):
    """
    Collect messages in msg_names for duration_s seconds,
    then compute average Hz/jitter based only on wall-clock time.
    """
    end_t = time.time() + duration_s

    wall_times = defaultdict(list)  # {name: [t_wall, ...]}

    # Buffer warm-up: clear remaining old messages
    flushed = 0
    while True:
        m = master.recv_match(blocking=False)
        if not m:
            break
        flushed += 1
    if flushed:
        print(f"(Warm-up) Cleared {flushed} remaining messages")

    last_verbose = time.time()
    while time.time() < end_t:
        msg = master.recv_match(blocking=True, timeout=0.5)
        if not msg:
            continue

        name = msg.get_type()
        if name not in msg_names:
            continue

        now = time.time()
        wall_times[name].append(now)

        if verbose_every and (now - last_verbose) >= verbose_every:
            last_verbose = now
            print(
                f"[{time.strftime('%H:%M:%S')}] Collecting... "
                + ", ".join(f"{k}:{len(v)}" for k, v in wall_times.items())
            )

    def calc_stats(ts_list):
        """Continuous receive timestamp list -> average Hz, min/max/avg interval, count"""
        if len(ts_list) < 2:
            return dict(
                count=len(ts_list),
                hz=None,
                dt_min=None,
                dt_max=None,
                dt_avg=None,
                dt_std=None,
            )

        dts = [t2 - t1 for t1, t2 in zip(ts_list[:-1], ts_list[1:])]
        span = ts_list[-1] - ts_list[0]
        hz = (len(ts_list) - 1) / span if span > 0 else None

        return dict(
            count=len(ts_list),
            hz=hz,
            dt_min=min(dts),
            dt_max=max(dts),
            dt_avg=sum(dts) / len(dts),
            dt_std=stats.pstdev(dts) if len(dts) > 1 else 0.0,
        )

    result = {}
    for name in msg_names:
        result[name] = calc_stats(wall_times.get(name, []))

    return result


# Jupyter cell 3
import pandas as pd

def to_dataframe(result):
    rows = []
    for name, wall in result.items():
        rows.append({
            "msg_name": name,
            "count": wall["count"],
            "Hz": wall["hz"],
            "dt_avg": wall["dt_avg"],
            "dt_min": wall["dt_min"],
            "dt_max": wall["dt_max"],
            "dt_std": wall["dt_std"],
        })

    df = pd.DataFrame(rows)

    with pd.option_context('display.float_format', lambda x: f"{x:.4f}" if isinstance(x, float) else x):
        display(df)

    return df


# Jupyter cell 4
DURATION = 20.0  # seconds

print(f"Measurement started: {DURATION} s")
res = measure_rates(duration_s=DURATION, msg_names=MSG_NAMES, verbose_every=5)
df = to_dataframe(res)

print("\nInterpretation guide:")
print("- Hz     : Actual receive rate on the offboard side (this notebook), based on wall-clock time.")
print("- dt_avg : Average interval between consecutive received messages.")
print("- dt_std : Jitter of receive interval.")
print("- dt_min/max : Minimum/maximum interval observed.")